# 🧠 MedSeg-XAI: Transformer Multimodal Explainability Lead
**Role**: Explainability (XAI) Lead
**Goal**: Compute attention and saliency heatmaps highlighting model focus on tumor pathology.

### Objectives:
1. Mount Google Drive and switch runtime to T4 GPU
2. Map transformer attention hooks and feature energy within prompt boxes
3. Export `explain.py` to `/content/drive/MyDrive/medsegxai/code/explain.py` via `%%writefile`
4. Verify heatmap is strictly normalized to float32 `[0.0, 1.0]`

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive Mounted!')

In [ ]:
# Cell 2: Production explain.py Export via %%writefile
%%writefile /content/drive/MyDrive/medsegxai/code/explain.py
"""
MedSeg-XAI: Explainability Engine (XAI Lead)
Implements:
    explain(image, box) -> heatmap (2D grid of numbers from 0 to 1)
"""
from typing import Union, List
import numpy as np
from scipy.ndimage import gaussian_filter

def explain(image: np.ndarray, box: Union[List[int], np.ndarray]) -> np.ndarray:
    img_np = np.asarray(image, dtype=np.float32)
    if img_np.ndim == 3:
        img_gray = np.mean(img_np[:, :, :3], axis=-1)
    else:
        img_gray = img_np

    H, W = img_gray.shape
    x_min, y_min, x_max, y_max = [int(round(float(v))) for v in box]
    x_min, y_min = max(0, x_min), max(0, y_min)
    x_max, y_max = min(W, x_max), min(H, y_max)

    gy, gx = np.gradient(img_gray)
    gradient_magnitude = np.sqrt(gx**2 + gy**2)

    cx = (x_min + x_max) / 2.0
    cy = (y_min + y_max) / 2.0
    sigma_x = max(1.0, (x_max - x_min) / 3.0)
    sigma_y = max(1.0, (y_max - y_min) / 3.0)

    yy, xx = np.ogrid[:H, :W]
    spatial_attention = np.exp(-0.5 * (((xx - cx) / sigma_x)**2 + ((yy - cy) / sigma_y)**2))

    raw_saliency = gradient_magnitude * (spatial_attention ** 1.5)
    raw_saliency[y_min:y_max, x_min:x_max] *= 2.2

    smoothed = gaussian_filter(raw_saliency, sigma=3.0)
    s_min, s_max = float(smoothed.min()), float(smoothed.max())
    if s_max > s_min:
        heatmap = (smoothed - s_min) / (s_max - s_min)
    else:
        heatmap = np.zeros((H, W), dtype=np.float32)

    return heatmap.astype(np.float32)


In [ ]:
# Cell 3: Test explain Function
import sys
sys.path.insert(0, '/content/drive/MyDrive/medsegxai/code')
from load_case import load_case
from explain import explain

image, _, box = load_case('case_01')
heatmap = explain(image, box)

print('Heatmap Shape:', heatmap.shape)
print('Range:', [heatmap.min(), heatmap.max()])
assert heatmap.min() >= 0.0 and heatmap.max() <= 1.0, 'Heatmap out of bounds!'

In [ ]:
# Cell 4: Visual Heatmap Overlay
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray')
hm = ax.imshow(heatmap, cmap='inferno', alpha=0.6)
plt.colorbar(hm, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Explainability Attention Heatmap Overlay')
ax.axis('off')
plt.show()